# Практика 4. Отчётность компаний: RFSD
**Количественные финансы · НИУ ВШЭ · 26 ноября 2026**

Сегодня работаем с бухгалтерской отчётностью российских компаний из набора RFSD. Вопрос занятия: **что можно узнать об отрасли по открытой отчётности и насколько этим данным можно доверять?** На паре работаем с выборкой книги, дома – с полным RFSD: домашняя часть – начало проекта 1 «Фундаментальный анализ отрасли».

Материал – [глава 5 «Большие данные корпоративной отчётности»](https://quantviews.github.io/quant-finance-book/05-accounting-data.html).

| Когда | Задания |
|---|---|
| Вторая пара · 60 минут | 0–5 |
| Дома: начало проекта 1 | 6–8 и всё, что не успели |

Сдаём выполненный ноутбук `practice04_фамилия.ipynb` с ответами в ячейках «**Ответ:**» способом, который объявит преподаватель.

Если застряли больше чем на 10 минут, запишите в ответ, что пробовали, текст ошибки и вопрос, и переходите к следующему заданию.

**Имя и группа:** …

**Моя акция (тикер):** …

**Мой класс ОКВЭД:** …

### Что вписать: тикер и класс
- **Тикер** – короткий код акции на Московской бирже: `GAZP` – «Газпром», `AFLT` – «Аэрофлот». Берите акцию, которая досталась вам в самостоятельной работе 2 ([таблица распределения](https://docs.google.com/spreadsheets/d/1jDm3lILAcKnvIehthge5Q3NfrHSzqmem/edit?usp=sharing)) – ту же, что на занятиях 2 и 3. Для обыкновенных и привилегированных акций тикеры разные: пишите так, как в таблице распределения (`SBERP`, `TRNFP`).
- **Класс ОКВЭД** вы не выбираете: найдите свой тикер в правом столбце таблицы ниже, ваш класс – в левом. Это учебная отрасль для заданий 3–5, по пять студентов на класс. С отраслью вашей акции она не связана. Ячейка с тикером ниже определит класс сама.
- **ИНН эмитента** искать не нужно: в файле эмитентов каждому тикеру уже сопоставлен ИНН головной компании (см. «Что в данных»).

| Класс ОКВЭД | Тикеры |
|---|---|
| 10 производство пищевых продуктов | AFLT, GAZP, MAGN, PHOR, SBERP |
| 41 строительство зданий | ALRS, GMKN, MOEX, PIKK, SIBN |
| 47 розничная торговля | BSPB, HYDR, MTSS, PLZL, SNGS |
| 49 сухопутный и трубопроводный транспорт | CHMF, IRAO, NLMK, ROSN, TATN |
| 62 разработка программного обеспечения | FEES, LKOH, NVTK, RUAL, TRNFP |

## 0. Подготовка
Ячейки ниже подключают библиотеки и загружают два файла: выборку RFSD из книги (8 МБ) и отчётность эмитентов акций курса. Выполните их один раз, редактировать не нужно. Без сети ноутбук читает файлы из папки `data` рядом с собой.

Нужна библиотека **Polars**. В Google Colab она уже установлена; в своём Jupyter выполните `pip install polars pyarrow`.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.3f}".format
pl.Config.set_tbl_rows(20)
print("Библиотеки загружены, polars", pl.__version__)

In [ ]:
# Загрузка данных: сначала с GitHub, без сети – из папки data рядом с ноутбуком.
import hashlib, io, urllib.request   # urllib – и для полного RFSD в задании 6
from pathlib import Path

def fetch(url, local, sha):
    local = Path(local)
    try:
        raw = urllib.request.urlopen(url, timeout=30).read()
        local.parent.mkdir(exist_ok=True)
        local.write_bytes(raw)                 # копия для Polars и для работы без сети
        print("Загружено с GitHub:", local.name)
    except OSError:
        raw = local.read_bytes()
        print("Нет доступа к GitHub, прочитан файл", local)
    assert hashlib.sha256(raw).hexdigest() == sha, f"{local.name} отличается от снимка занятия"
    return raw

SAMPLE = Path("data/rfsd_sample.parquet")
fetch("https://raw.githubusercontent.com/quantviews/quant-finance-book/6a42857136078f2a01d5e0d85c9d9cf211b14fce/data/rfsd_sample.parquet", SAMPLE, "9b0171a181556c0df0db07397f49a2017b2f25b2108403318f95394203c7557a")
raw = fetch("https://raw.githubusercontent.com/quantviews/quant-finance-book/main/lessons/04-accounting-data/data/issuers_2018_2025.csv", "data/issuers_2018_2025.csv", "cfbb64e1130adcad8070989d57e02633ee4b268c3149ed975436f26e2c9f7b5a")

rfsd = pd.read_parquet(SAMPLE)                                     # pandas: вся таблица в памяти
issuers = pd.read_csv(io.BytesIO(raw), dtype={"inn": str})
print("Выборка:", rfsd.shape, "· годы", rfsd["year"].min(), "–", rfsd["year"].max())
print("Эмитенты:", issuers["ticker"].nunique(), "тикеров")

### Что в данных
**Выборка книги** `rfsd_sample.parquet`: компании пяти классов ОКВЭД (10, 41, 47, 49, 62), выручка которых хотя бы в одном году 2018–2025 превышала 1 млрд руб. Одна строка – отчёт компании за год. `name` – название, добавлено отдельно (в RFSD названий нет).

**Эмитенты** `issuers_2018_2025.csv`: отчётность головных компаний-эмитентов акций курса за 2018–2025 гг. из полного RFSD. Пустые строки – отчётность не раскрыта или организации нет в RFSD.

Соглашения RFSD:
- денежные показатели – **в тысячах рублей**;
- строки, которые в формах пишутся в скобках (себестоимость, проценты к уплате), хранятся **отрицательными**;
- `filed = 1` – отчёт подан; `imputed = 1` – значения восстановлены; `outlier = 1` – подозрительный отчёт по оценке авторов RFSD.

### ИНН и ОКВЭД: что это и откуда
**ИНН** – номер организации в налоговой, 10 цифр. Отчётность сдаёт юридическое лицо, поэтому в RFSD одна компания – это один ИНН. По ИНН таблицы соединяются друг с другом, с ГИР БО и с реестром юрлиц.

- У компании, которую мы знаем по бренду, обычно много юрлиц и много ИНН: «Магнит» – это ПАО «Магнит» (держит акции) и АО «Тандер» (магазины).
- **Эмитент** – юрлицо, которое выпустило акцию. Его ИНН указан в карточке компании в [центре раскрытия информации](https://www.e-disclosure.ru) и на сайте компании в разделе «Реквизиты» или «Раскрытие информации». Для акций курса эти ИНН уже собраны в файле эмитентов.
- Проверить ИНН и посмотреть отчётность по нему можно в [ГИР БО](https://bo.nalog.gov.ru): поиск по названию или ИНН.

**ОКВЭД** – классификатор видов экономической деятельности. Код читается слева направо, от общего к частному:

| Уровень | Пример | Что означает |
|---|---|---|
| класс | `10` | производство пищевых продуктов |
| подкласс | `10.5` | производство молочной продукции |
| группа | `10.51` | производство молока (кроме сырого) и молочной продукции |
| подгруппа | `10.51.3` | производство сыра и сырных продуктов |

- В RFSD `okved` – **основной** код, который компания сама указала в реестре юрлиц. Он не всегда совпадает с тем, чем компания известна: у «Газпрома» – `46.71`, оптовая торговля топливом.
- Отбор по началу кода берёт все уровни ниже: `str.starts_with("10")` – весь класс 10, `"10.51"` – только группу.
- Названия кодов – в [классификаторе rustata](https://rustata.ru/classifier/okved2).

Источник: RFSD (Bondarkov, Ledenev, Skougarevskiy, 2025), CC BY 4.0; снимок RFSD v3 от 02.09.2026.

In [ ]:
TICKER = "..."       # ваша акция из самостоятельной работы 2, например "GAZP"

# Класс ОКВЭД для заданий 3–5 – по таблице в начале ноутбука, вписывать не нужно
TICKERS_BY_CLASS = {
    "10": ["AFLT", "GAZP", "MAGN", "PHOR", "SBERP"],  # производство пищевых продуктов
    "41": ["ALRS", "GMKN", "MOEX", "PIKK", "SIBN"],  # строительство зданий
    "47": ["BSPB", "HYDR", "MTSS", "PLZL", "SNGS"],  # розничная торговля
    "49": ["CHMF", "IRAO", "NLMK", "ROSN", "TATN"],  # сухопутный и трубопроводный транспорт
    "62": ["FEES", "LKOH", "NVTK", "RUAL", "TRNFP"],  # разработка программного обеспечения
}
MY_CLASS = next((k for k, v in TICKERS_BY_CLASS.items() if TICKER in v), None)
assert MY_CLASS, "Такого тикера нет в таблице – проверьте написание: заглавные латинские буквы"

rfsd["div"] = rfsd["okved"].str[:2]
pos = lambda x: x.where(x > 0)          # знаменатель: только положительные значения
print(TICKER, "· класс ОКВЭД", MY_CLASS)

## Задание 1. Эмитент вашей акции · 10 минут
1. За какие годы 2018–2025 отчётность эмитента вашей акции раскрыта (`filed = 1`)? Какой у него код ОКВЭД?
2. Если за 2025 г. отчётности нет, объясните почему: банк или биржа (отчитываются перед Банком России), скрытая отчётность или другое. С какого года её не видно?
3. Если отчётность есть: выручка, доходы от участия (строка 2310) и чистая прибыль за последний раскрытый год в млрд руб. Это операционная компания или холдинг? По каким числам вы это видите?
4. Найдите в раскрытии эмитента выручку **группы** по МСФО за тот же год (сайт компании, раздел для инвесторов). Во сколько раз она отличается от выручки по РСБУ и почему? *(Можно дома.)*

**Ответ:** …

In [ ]:
cols = ["year", "okved", "filed", "line_2110", "line_2310", "line_2400", "line_1300", "line_1600"]
me = issuers[issuers["ticker"] == TICKER]
print(me["issuer"].iloc[0], "· ИНН", me["inn"].iloc[0])
me[cols]
# Переведите тыс. руб. в млрд руб.: деление на 1e6


## Задание 2. Единицы и знаки · 10 минут
АО «Тандер» (ИНН 2310031475) – операционная компания сети «Магнит».

**Сначала предскажите:** выручка «Тандера» за 2025 г. – это сотни миллиардов или триллионы рублей? Получила ли компания чистую прибыль?

1. Выручка, себестоимость, валовая прибыль (`line_2110 + line_2120`) и чистая прибыль «Тандера» за 2025 г. в млрд руб.
2. Чему была бы равна «валовая прибыль», если бы вы посчитали её как `line_2110 − line_2120`? Во сколько раз она завышена?
3. EBIT = `line_2300 − line_2330` и покрытие процентов = EBIT / (−`line_2330`). Как изменились проценты к уплате и покрытие с 2023 по 2025 г.? Что это говорит о долговой нагрузке компании?

**Предсказание:** …

**Ответ:** …

In [ ]:
td = rfsd[rfsd["inn"] == "2310031475"].set_index("year")
td[["line_2110", "line_2120", "line_2300", "line_2330", "line_2400"]] / 1e6   # млрд руб.
# Посчитайте валовую прибыль, EBIT и покрытие процентов по годам


## Задание 3. Parquet, CSV, pandas и Polars · 15 минут
1. Сохраните выборку в CSV (код ниже). Сравните размеры файлов CSV и parquet и время их чтения в pandas. Затем прочитайте из parquet только 5 столбцов (аргумент `columns=`). Во сколько раз это быстрее чтения всей таблицы?
2. Найдите ОАО «Молзавод „Гиагинский“» (ИНН `0101000776`) в таблице, прочитанной из CSV, и в таблице из parquet. Что не так с CSV и чем это опасно, когда таблицы соединяют по ИНН?
3. Постройте **ленивый** запрос Polars к `data/rfsd_sample.parquet`: поданные отчёты за 2025 г. вашего класса ОКВЭД, 10 крупнейших компаний по выручке (название, ИНН, код ОКВЭД, выручка и чистая прибыль в млрд руб.). Не забудьте `nulls_last=True`. Выведите план методом `.explain()`: на каком шаге применяется фильтр и сколько столбцов читается из файла?
4. Сколько компаний класса подали отчётность за 2025 г.? Какую долю выручки класса дают 10 крупнейших?

**Ответ:** …

In [ ]:
import time
CSV = Path("data/rfsd_sample.csv")
rfsd.drop(columns="div").to_csv(CSV, index=False)
print(f"CSV: {CSV.stat().st_size / 1e6:.1f} МБ · parquet: {SAMPLE.stat().st_size / 1e6:.1f} МБ")

t = time.perf_counter(); from_csv = pd.read_csv(CSV, low_memory=False); t_csv = time.perf_counter() - t
t = time.perf_counter(); from_pq = pd.read_parquet(SAMPLE); t_pq = time.perf_counter() - t
print(f"Чтение в pandas: CSV {t_csv:.2f} с, parquet {t_pq:.2f} с")
# Допишите: чтение 5 столбцов из parquet; поиск ИНН "0101000776" в from_csv и from_pq; from_csv["inn"].dtype


In [ ]:
lf = pl.scan_parquet(SAMPLE)          # ничего не прочитано
q = (lf.filter((pl.col("year") == 2025) & (pl.col("filed") == 1)
               & pl.col("okved").str.starts_with(MY_CLASS))
       .select("name", "inn", "okved", "line_2110", "line_2400"))
# Допишите: сортировка, head(10), столбцы в млрд руб.; затем q.explain() и q.collect()


## Задание 4. Качество данных вашего класса · 15 минут
Для поданных отчётов вашего класса за 2025 г. посчитайте:

1. Долю отчётов, где не сходится баланс: |`line_1600` − `line_1700`| > 5 тыс. руб.
2. Долю отчётов с неположительным капиталом (`line_1300` ≤ 0).
3. Долю отчётов, где деньги в балансе (`line_1250`) и в отчёте о движении денежных средств (`line_4500`) расходятся больше чем на 5 тыс. руб. (среди отчётов, где есть обе строки).
4. Долю компаний класса, у которых основной код ОКВЭД менялся за 2018–2025 гг.

Какая из проблем важнее всего для анализа рентабельности в вашем классе и почему?

**Ответ:** …

In [ ]:
d = rfsd[(rfsd["year"] == 2025) & (rfsd["filed"] == 1) & (rfsd["div"] == MY_CLASS)].copy()
print("Отчётов:", len(d))
mismatch = (d["line_1600"] - d["line_1700"]).abs() > 5
print(f"Баланс не сходится: {mismatch.mean():.2%}")
# Допишите остальные проверки


## Задание 5. Коэффициенты: медиана и среднее · 15 минут
**Сначала предскажите:** средняя ROE в вашем классе выше или ниже медианной? Во сколько раз?

1. Посчитайте для всех пяти классов за 2025 г.: ROS, ROA, ROE, оборачиваемость активов, текущую ликвидность (формулы – на слайде и в таблице главы 5). Знаменатель – только положительный: функция `pos`.
2. Таблица медиан по классам. Где ваш класс выше остальных, где ниже? Объясните хотя бы одно различие бизнес-моделью.
3. Для своего класса: медиана, квартили (0,25 и 0,75) и среднее ROE. Почему среднее так далеко от медианы? Найдите компанию с наибольшей ROE и объясните, откуда такое значение.

**Предсказание:** …

**Ответ:** …

In [ ]:
a = rfsd[(rfsd["year"] == 2025) & (rfsd["filed"] == 1)].copy()
a["ros"] = a["line_2400"] / pos(a["line_2110"])
a["roe"] = a["line_2400"] / pos(a["line_1300"])
# Допишите roa, turnover, current; затем a.groupby("div")[[...]].median()


## Задание 6. Проект 1: кандидаты в тему по полному RFSD · дома, 40 минут
Выборка книги – только пять классов и компании с выручкой от 1 млрд руб. Для проекта нужен полный RFSD. Это задание – первый шаг проекта 1: вы проверяете, хватает ли данных для темы.

Выберите **2–3 кандидата**: отрасль (код ОКВЭД любого уровня, например `10.51` – молочная продукция) или отрасль в одном регионе.

**Как найти код ОКВЭД отрасли.** Два пути, лучше пройти оба:
- **От отрасли к коду.** Пройдите по дереву [классификатора rustata](https://rustata.ru/classifier/okved2) от класса вниз: `10` → `10.5` → `10.51`. Выбирайте группу или подгруппу (4–5 знаков): класс из двух цифр слишком широкий – `10` объединяет и молоко, и мясо, и хлеб.
- **От компании к коду.** Возьмите известную компанию отрасли, найдите её в [ГИР БО](https://bo.nalog.gov.ru) по названию: в карточке будут ИНН и основной код ОКВЭД.

Проверка кода: посмотрите 5 крупнейших компаний кандидата (код ниже). Если это не те компании, которые вы ожидали, код выбран неверно или лидеры записаны под другим кодом.

**Как найти ИНН компании.** ГИР БО: поиск по названию. Другие источники – [реестр юрлиц ФНС](https://egrul.nalog.ru) и раздел «Реквизиты» или «Контакты» на сайте компании. У бренда часто несколько юрлиц: берите то, у которого основная выручка (для «Магнита» – АО «Тандер», а не ПАО «Магнит»). Если у лидера в RFSD другой код ОКВЭД, он не попадёт в отбор по коду – это и есть **скрытый лидер**.

Для каждого кандидата за 2025 г. (поданные отчёты):

1. Сколько компаний и сколько из них с выручкой от 100 млн руб.? Суммарная выручка, доля 5 крупнейших.
2. **Охват:** сравните суммарную выручку в RFSD с выручкой организаций по данным Росстата для того же кода (страница кода в классификаторе rustata). Какую часть отрасли вы видите?
3. **Скрытые лидеры:** назовите 2–3 крупнейшие компании отрасли, которые вы знаете (сайты компаний, рейтинги, новости), найдите их ИНН и проверьте, есть ли их отчёт за 2025 г.
4. Какой кандидат проходит правило проекта (не меньше 30 компаний с выручкой от 100 млн руб.) и в каком данных больше? Сохраните таблицу выбранной темы в parquet.

**Ответ:** …

### Два способа читать полный RFSD
Файл RFSD за 2025 г. – 534 МБ, 3,14 млн строк и 223 столбца.

- **Б. Скачать файл один раз** (`USE_DOWNLOAD = True`, по умолчанию). Скачивание занимает 1–3 минуты, потом запросы быстрые и не требуют сети. В Colab файл пропадает после закрытия сессии – скачайте заново или сохраните на Google Drive.
- **А. Читать прямо с Hugging Face** (`USE_DOWNLOAD = False`). Polars скачивает только нужные столбцы, но при каждом запуске заново.

Ссылка закреплена на версию RFSD, с которой работает курс: ответы не изменятся, если авторы обновят набор.

**Если Hugging Face не открывается из вашей сети** (так бывает в российских сетях), выполните это задание в Google Colab: он работает на серверах Google.

В полном RFSD **нет названий компаний**: ищите их по ИНН в [ГИР БО](https://bo.nalog.gov.ru). Нет и столбца `year`: год – в имени файла. Регионы записаны латиницей: `moscow city`, `moscow reg.`, `tatarstan`….

In [ ]:
RFSD_REV = "bde9b8b129a8be582b66de4e30ed79c9187a8515"                    # версия RFSD v3
PART = f"RFSD/year=2025/part-0.parquet"
USE_DOWNLOAD = True                                 # False – читать прямо с Hugging Face

if USE_DOWNLOAD:
    source = Path(f"data/rfsd_2025.parquet")
    if not source.exists():
        import shutil
        source.parent.mkdir(exist_ok=True)
        tmp = source.with_suffix(".part")
        print("Скачиваю", PART, "– 1–3 минуты…")
        try:
            url = f"https://huggingface.co/datasets/irlspbru/RFSD/resolve/{RFSD_REV}/{PART}"
            with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as out:
                shutil.copyfileobj(r, out, 1 << 20)
        except OSError as e:
            raise RuntimeError("Hugging Face не отвечает из вашей сети. Откройте ноутбук в Google Colab "
                               "или скачайте файл по ссылке выше в браузере и положите в data/") from e
        tmp.rename(source)                          # недокачанный файл не останется под рабочим именем
    print(f"{source}: {source.stat().st_size / 1e6:.0f} МБ")
else:
    source = f"hf://datasets/irlspbru/RFSD@{RFSD_REV}/{PART}"

full = pl.scan_parquet(source)
print(len(full.collect_schema()), "столбцов")

In [ ]:
COLS = ["inn", "okved", "region", "line_2110", "line_2120", "line_2200", "line_2300", "line_2330",
        "line_2400", "line_1300", "line_1600", "line_1410", "line_1510"]

def candidate(okved, region=None):
    """Поданные отчёты по коду ОКВЭД (и региону) – таблица Polars."""
    cond = (pl.col("filed") == 1) & pl.col("okved").str.starts_with(okved)
    if region:
        cond = cond & (pl.col("region") == region)
    return full.filter(cond).select(COLS).collect()

def summary(df):
    rev = df["line_2110"].fill_null(0).sort(descending=True)
    return {"компаний": df.height,
            "от 100 млн руб.": int((rev >= 100_000).sum()),
            "выручка, млрд руб.": round(rev.sum() / 1e6, 1),
            "доля топ-5": round(rev.head(5).sum() / rev.sum(), 3) if rev.sum() > 0 else None}

# Ваши кандидаты: название -> (код ОКВЭД, регион или None)
CANDIDATES = {"Молочная продукция": ("10.51", None),
              "...": ("...", None)}
tables = {name: candidate(*args) for name, args in CANDIDATES.items()}
pd.DataFrame.from_dict({name: summary(df) for name, df in tables.items()}, orient="index")

In [ ]:
# Регионы в RFSD: full.select("region").unique().sort("region").collect()
# 5 крупнейших компаний кандидата:
# tables["Молочная продукция"].sort("line_2110", descending=True, nulls_last=True).head(5)

# Скрытые лидеры: впишите ИНН известных компаний отрасли (10 цифр, в кавычках – это текст)
LEADERS = ["..."]
full.filter(pl.col("inn").is_in(LEADERS)).select("inn", "okved", "region", "filed", "line_2110").collect()

# Таблица выбранной темы – для проекта:
# tables["..."].write_parquet("data/project1.parquet")


## Задание 7. Проверка кода ИИ-ассистента · дома, 10 минут
Ассистента попросили «посчитать по классам ОКВЭД валовую прибыль, ROE и отношение долга к EBITDA». Код ниже работает без ошибок, но результат неверен.

Найдите не меньше четырёх ошибок, объясните каждую и напишите исправленный код. Сравните результаты до и после.

**Ответ:** …

In [ ]:
# Код ассистента – не меняйте эту ячейку, исправленный вариант напишите ниже
x = rfsd[rfsd["year"] == 2025].copy()
x["gross"] = x["line_2110"] - x["line_2120"]
x["ebitda"] = x["line_2300"] + x["line_2330"]
x["roe"] = x["line_2400"] / x["line_1300"]
x["debt_to_ebitda"] = (x["line_1410"] + x["line_1510"]) / x["ebitda"]
print(x.groupby(x["okved"].str[:2])[["gross", "roe", "debt_to_ebitda"]].mean())

In [ ]:
# Исправленный код


## Задание 8. Заявка на проект 1 · дома, 150–250 слов
Это заявка на тему проекта 1. Тему также впишите в таблицу курса: темы не повторяются, преподаватель подтверждает заявку.

1. **Тема** и **правило отбора компаний**: коды ОКВЭД, регион, год, порог выручки, `filed = 1`.
2. **Почему эта тема:** число компаний, доля пяти крупнейших, охват по Росстату и скрытые лидеры – ваши числа из задания 6. Почему не другие кандидаты?
3. **Гипотеза** к вопросу проекта «за счёт чего лидеры отрасли эффективнее остальных – маржи, оборачиваемости или долговой нагрузки?». Запишите её **до** расчётов и объясните, на чём она основана.
4. Два главных **ограничения данных** для вашей темы и как каждое может сдвинуть вывод.

Если пользовались ИИ, запишите: **что спросили**, **что получили**, **как проверили**.

**Заявка:** …

**Журнал ИИ:** …

## Перед сдачей
- Перезапустите ядро и выполните все ячейки сверху вниз.
- Проверьте, что в каждом задании заполнена ячейка «**Ответ:**».
- Сохраните файл `practice04_фамилия.ipynb` и сдайте его способом, который объявит преподаватель.

Справочник строк отчётности – `rfsd_lines.csv` в [данных книги](https://github.com/quantviews/quant-finance-book/tree/main/data); синтаксис Polars – в [главе 5](https://quantviews.github.io/quant-finance-book/05-accounting-data.html#sec-polars).